# PySpark E-Commerce Data Processing Pipeline

This notebook implements a distributed data processing pipeline using PySpark, featuring **Interactive Visualizations**.

### Pipeline Steps:
1.  **Initialize**: Setup SparkSession and environment.
2.  **Load Data**: Ingest synthetic transaction data.
3.  **Data Quality**: Validate and clean data.
4.  **Analysis**: Filter high-value transactions, aggregate sales, and compute rolling averages.
5.  **Advanced Features**: Customer Segmentation (Rankings) and Business SQL Queries.
6.  **Interactive EDA**: Visualize distributions and trends using Plotly.
7.  **Machine Learning**: Train Regression and Deep Learning models.
8.  **Export**: Save results to CSV.

### Expected Input Schema:
- `transaction_id`: Unique identifier
- `category`: Product category
- `amount`: Transaction value
- `timestamp`: Time of transaction `YYYY-MM-DD HH:MM:SS`

In [81]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, desc, to_date, avg, round as spark_round, rank, when
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator, MulticlassClassificationEvaluator
import os
import sys

# Set HADOOP_HOME to local hadoop dir with winutils.exe to avoid Windows errors
os.environ['HADOOP_HOME'] = os.path.join(os.getcwd(), 'hadoop')

# Initialize Spark Session
spark = SparkSession.builder \
    .appName("EcommerceTransactionProcessing") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to reduce noise (hides BLAS/LAPACK warnings)
spark.sparkContext.setLogLevel("ERROR")

print("Spark Session created successfully.")

Spark Session created successfully.


## 1. Load Data
Load data from `data/transactions.csv` and infer the schema.

In [82]:
# Use absolute path with file:/// protocol to ensure local file system is used correctly on Windows
input_path = "file:///" + os.path.join(os.getcwd(), "data", "transactions_20k.csv").replace("\\", "/")
print(f"Loading data from {input_path}...")

# Infer schema to handle data types correctly
df = spark.read.csv(input_path, header=True, inferSchema=True)

print(f"Total records loaded: {df.count()}")
df.printSchema()

Loading data from file:///c:/Users/umesh/Documents/Programs/data/transactions_20k.csv...
Total records loaded: 20000
root
 |-- transaction_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)



## 2. Data Quality Check
Remove rows with nulls in critical columns or invalid amounts (Example: <= 0).

In [83]:
print("--- Data Quality Check ---")
initial_count = df.count()

# Remove rows with nulls in critical columns or negative amounts
df_clean = df.dropna(subset=["transaction_id", "user_id", "amount", "timestamp"]) \
       .filter(col("amount") > 0)

clean_count = df_clean.count()
print(f"Invalid records removed: {initial_count - clean_count}")
print(f"Clean records remaining: {clean_count}")

# Update main dataframe reference
df = df_clean

--- Data Quality Check ---
Invalid records removed: 0
Clean records remaining: 20000


## 3. High Value Transactions
Filter for transactions greater than $500.

In [84]:
print("--- Filtering High Value Transactions (> $500) ---")
high_value_df = df.filter(col("amount") > 500)
print(f"High value transaction count: {high_value_df.count()}")
high_value_df.show(5)

--- Filtering High Value Transactions (> $500) ---
High value transaction count: 3069
+--------------------+-------+-----------+-------+-------------------+
|      transaction_id|user_id|   category| amount|          timestamp|
+--------------------+-------+-----------+-------+-------------------+
|c53dbcb7-a91c-4cc...|     40|Electronics|1692.04|2024-02-22 19:08:29|
|25a77f5e-415c-46e...|    799|Electronics|1833.45|2024-09-27 23:04:43|
|ec04bfc3-2e45-424...|    918|Electronics| 922.29|2024-01-20 06:35:22|
|6db19137-d09e-43b...|    256|Electronics|1478.01|2024-10-27 13:11:24|
|128512ed-7d32-416...|    431|Electronics|1553.63|2024-07-27 23:27:55|
+--------------------+-------+-----------+-------+-------------------+
only showing top 5 rows



## 4. Sales Aggregation by Category

In [85]:
print("--- Aggregating Total Sales by Category ---")
category_summary = df.groupBy("category") \
    .agg(
        spark_sum("amount").alias("total_sales"),
        count("transaction_id").alias("transaction_count")
    ) \
    .orderBy(desc("total_sales"))

category_summary.show()

--- Aggregating Total Sales by Category ---
+-----------+-----------------+-----------------+
|   category|      total_sales|transaction_count|
+-----------+-----------------+-----------------+
|Electronics|4076098.619999989|             3982|
|     Sports|423486.9499999993|             4030|
|      Books|419859.5599999999|             3997|
|   Clothing|419138.2899999997|             3978|
|       Home|418065.1700000004|             4013|
+-----------+-----------------+-----------------+



## 5. Time-Based Analysis
Calculate daily sales and a **7-Day Rolling Average** per category.

In [86]:
print("--- Time-Based Analysis: Daily Sales & 7-Day Rolling Avg (Per Category) ---")

# Convert timestamp to date type
df_dates = df.withColumn("date", to_date(col("timestamp")))

# Aggregation: Daily Sales Per Category
daily_sales = df_dates.groupBy("date", "category") \
    .agg(spark_sum("amount").alias("daily_total"))
    
# Window Function: 7-Day Rolling Average PER CATEGORY
# Partitions by category so operations are distributed (avoiding single-partition performance usage)
w = Window.partitionBy("category").orderBy("date").rowsBetween(-6, 0)

daily_trends = daily_sales.withColumn("7_day_avg", spark_round(avg("daily_total").over(w), 2)) \
    .orderBy("category", "date")
    
daily_trends.show(10)

--- Time-Based Analysis: Daily Sales & 7-Day Rolling Avg (Per Category) ---
+----------+--------+------------------+---------+
|      date|category|       daily_total|7_day_avg|
+----------+--------+------------------+---------+
|2024-01-01|   Books| 768.0899999999999|   768.09|
|2024-01-02|   Books|            662.05|   715.07|
|2024-01-03|   Books| 822.5699999999999|    750.9|
|2024-01-04|   Books|           1565.22|   954.48|
|2024-01-05|   Books|           1815.06|   1126.6|
|2024-01-06|   Books|1441.6799999999998|  1179.11|
|2024-01-07|   Books|1399.6499999999999|  1210.62|
|2024-01-08|   Books|            971.22|  1239.64|
|2024-01-09|   Books|            896.28|   1273.1|
|2024-01-10|   Books| 804.5899999999999|  1270.53|
+----------+--------+------------------+---------+
only showing top 10 rows



## 6. Customer Segmentation (Rankings)
Identify the **Top 3 Spenders** in each category using Window functions.

In [87]:
print("--- Customer Segmentation: Top 3 Spenders per Category ---")

# Calculate total spend per user per category
user_category_spend = df.groupBy("category", "user_id") \
    .agg(spark_sum("amount").alias("total_spend"))

# Rank users within each category
w_rank = Window.partitionBy("category").orderBy(desc("total_spend"))
ranked_users = user_category_spend.withColumn("rank", rank().over(w_rank))

# Filter for top 3
top_customers = ranked_users.filter(col("rank") <= 3)
top_customers.show(15) # Show 3 per category * 5 categories

--- Customer Segmentation: Top 3 Spenders per Category ---
+-----------+-------+------------------+----+
|   category|user_id|       total_spend|rank|
+-----------+-------+------------------+----+
|      Books|    142|           1197.52|   1|
|      Books|    376|1174.8500000000001|   2|
|      Books|    475|           1123.22|   3|
|   Clothing|    141|           1594.95|   1|
|   Clothing|    604|1338.7300000000002|   2|
|   Clothing|    763|1297.5700000000002|   3|
|Electronics|     49|14092.259999999998|   1|
|Electronics|    774|          13260.63|   2|
|Electronics|    557|11221.060000000001|   3|
|       Home|     86|           1397.43|   1|
|       Home|    485|1309.8599999999997|   2|
|       Home|    216|1241.6200000000001|   3|
|     Sports|    157|1228.4900000000002|   1|
|     Sports|    958|1203.1799999999998|   2|
|     Sports|    829|           1094.09|   3|
+-----------+-------+------------------+----+



## 7. Business SQL Queries
Strategic insights using Spark SQL.

In [ ]:
print("--- SQL Interface: Business Analytics Queries ---")

# Register DataFrame as a specific table
df.createOrReplaceTempView("transactions")

# Query 1: Peak Trading Hours
print("[SQL] Insight 1: Peak Transacting Hours (Total Transactions by Hour)")
sql_peak_hours = """
    SELECT 
        hour(timestamp) as hour_of_day, 
        count(*) as tx_count,
        round(sum(amount), 2) as hourly_revenue
    FROM transactions 
    GROUP BY hour(timestamp) 
    ORDER BY tx_count DESC
    LIMIT 5
"""
spark.sql(sql_peak_hours).show()

# Query 2: Category Performance Matrix
print("[SQL] Insight 2: Category Performance Matrix")
sql_cat_perf = """
    SELECT 
        category,
        round(sum(amount), 0) as total_revenue,
        round(avg(amount), 2) as avg_order_value,
        round(max(amount), 2) as max_transaction
    FROM transactions
    GROUP BY category
    ORDER BY total_revenue DESC
"""
spark.sql(sql_cat_perf).show()

--- SQL Interface: Business Analytics Queries ---
[SQL] Insight 1: Peak Trading Hours (Total Transactions by Hour)
+-----------+--------+--------------+
|hour_of_day|tx_count|hourly_revenue|
+-----------+--------+--------------+
|         17|     908|      275010.5|
|         10|     906|     269409.18|
|         20|     865|     245939.32|
|         12|     862|     264064.55|
|         14|     849|      277691.3|
+-----------+--------+--------------+

[SQL] Insight 2: Category Performance Matrix
+-----------+-------------+---------------+---------------+
|   category|total_revenue|avg_order_value|max_transaction|
+-----------+-------------+---------------+---------------+
|Electronics|    4076099.0|        1023.63|        1999.85|
|     Sports|     423487.0|         105.08|         199.92|
|      Books|     419860.0|         105.04|         199.97|
|   Clothing|     419138.0|         105.36|         199.98|
|       Home|     418065.0|         104.18|          200.0|
+-----------+----

## 8. Interactive Exploratory Data Analysis (Plotly)
Visualizing data using interactive charts with tooltips and zoom capabilities.

**Note:** Ensure `plotly` is installed: `pip install plotly`

### 8.1 Setup & Data Preparation

In [89]:
import plotly.express as px
import pandas as pd

print("--- Preparing Data for Visualization ---")

# Convert Spark DataFrame to Pandas for Plotly
# Collecting subset of columns needed for viz
pdf = df.select("category", "amount", "timestamp").toPandas()

# Feature Engineering for Plots
pdf["date"] = pd.to_datetime(pdf["timestamp"]).dt.date
pdf["hour"] = pd.to_datetime(pdf["timestamp"]).dt.hour
pdf["day_of_week"] = pd.to_datetime(pdf["timestamp"]).dt.day_name()

print(f"Data prepared: {pdf.shape[0]} rows ready for plotting.")

--- Preparing Data for Visualization ---
Data prepared: 20000 rows ready for plotting.


### 8.2 Distribution of Transaction Amounts
**Interactive Histogram**: Hover to see precise counts for each price bin.

In [90]:
fig = px.histogram(pdf, x="amount", nbins=50, title="Distribution of Transaction Amounts",
                   labels={'amount': 'Transaction Amount ($)'}, template="plotly_white")
fig.update_layout(bargap=0.1)
fig.show()

### 8.3 Category Insights (Box Plot)
**Interactive Box Plot**: Hover to see Median, Quartiles, and Outliers for each category.

In [91]:
fig = px.box(pdf, x="category", y="amount", color="category", 
             title="Transaction Amount Distribution by Category",
             template="plotly_white")
fig.show()

### 8.4 Daily Transaction Trends
**Interactive Line Chart**: Zoom in on specific time periods to see daily specific fluctuations.

In [92]:
daily_counts = pdf.groupby("date").size().reset_index(name="count")

fig = px.line(daily_counts, x="date", y="count", markers=True,
              title="Daily Transaction Volume",
              labels={'date': 'Date', 'count': 'Number of Transactions'},
              template="plotly_white")
fig.show()

### 8.5 Total Sales by Category
**Interactive Bar Chart**: Compare total revenue across categories.

In [93]:
category_sales = pdf.groupby("category")["amount"].sum().reset_index()

fig = px.bar(category_sales, x="category", y="amount", color="category",
             title="Total Sales Revenue by Category",
             text_auto='.2s',
             template="plotly_white")
fig.show()

### 8.6 Peak Activity Heatmap (Day vs. Hour)
**Interactive Heatmap**: Identify busiest trading hours during the week.

In [94]:
# Aggregate counts by Day and Hour
heatmap_data = pdf.groupby(["day_of_week", "hour"]).size().reset_index(name="count")

# Order days correctly
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig = px.density_heatmap(heatmap_data, x="hour", y="day_of_week", z="count", 
                         category_orders={"day_of_week": days_order},
                         title="Transaction Density Heatmap (Day vs Hour)",
                         color_continuous_scale="Viridis",
                         template="plotly_white")
fig.show()

## 9. Machine Learning & Deep Learning
Implement regression and classification models.

In [95]:
print("--- Machine Learning & Deep Learning Section ---")

# Feature Engineering
# 1. Convert Category string to Index (0, 1, 2...)
indexer = StringIndexer(inputCol="category", outputCol="categoryIndex")
df_ml = indexer.fit(df).transform(df)

# 2. Create Label for Classification (High Value > $200 = 1, else 0)
df_ml = df_ml.withColumn("high_value_label", when(col("amount") > 200, 1.0).otherwise(0.0))

# 3. Assemble Features Vector (using categoryIndex as the feature)
assembler = VectorAssembler(inputCols=["categoryIndex"], outputCol="features")
data = assembler.transform(df_ml)

# Split Data (80% Train, 20% Test)
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)
print(f"Training ML models on {train_data.count()} records...")

--- Machine Learning & Deep Learning Section ---
Training ML models on 16052 records...


### A1. Linear Regression: Training
Train the model using standard Train/Test split.

In [96]:
print("\n[ML] Training Linear Regression Model...")
# Initialize LR
lr = LinearRegression(featuresCol="features", labelCol="amount")

# Train Model
lr_model = lr.fit(train_data)


[ML] Training Linear Regression Model...


### A2. Linear Regression: Evaluation
Calculate RMSE, MAE, and R2 on Test Data.

In [97]:
lr_predictions = lr_model.transform(test_data)

# Define Evaluator (RMSE)
evaluator = RegressionEvaluator(labelCol="amount", predictionCol="prediction", metricName="rmse")

# Evaluation: RMSE
rmse_lr = evaluator.evaluate(lr_predictions)
print(f"Linear Regression RMSE: {rmse_lr:.2f}")

# Evaluation: MAE
mae_eval = RegressionEvaluator(labelCol="amount", predictionCol="prediction", metricName="mae")
mae_lr = mae_eval.evaluate(lr_predictions)
print(f"Linear Regression MAE: {mae_lr:.2f}")

# Evaluation: R2 Score
r2_eval = RegressionEvaluator(labelCol="amount", predictionCol="prediction", metricName="r2")
r2_lr = r2_eval.evaluate(lr_predictions)
print(f"Linear Regression R2 Score: {r2_lr:.4f}")

Linear Regression RMSE: 423.99
Linear Regression MAE: 276.07
Linear Regression R2 Score: 0.0789


### B1. Random Forest: Training
Train the Random Forest using standard Train/Test split.

In [98]:
print("\n[ML] Training Random Forest Regressor...")
# Use VectorIndexer to identify categorical features
featureIndexer = VectorIndexer(inputCol="features", outputCol="indexedFeatures", maxCategories=10).fit(data)

# Train RF with indexed features and fixed seed
rf = RandomForestRegressor(featuresCol="indexedFeatures", labelCol="amount", seed=42)

# Pipeline approach or just fit on transformed data
# Here we manually index the train/test data
train_indexed = featureIndexer.transform(train_data)
test_indexed = featureIndexer.transform(test_data)

rf_model = rf.fit(train_indexed)


[ML] Training Random Forest Regressor...


### B2. Random Forest: Evaluation
Calculate RMSE, MAE, and R2 on Test Data.

In [99]:
rf_predictions = rf_model.transform(test_indexed)

# Evaluation: RMSE
rmse_rf = evaluator.evaluate(rf_predictions)
print(f"Random Forest RMSE: {rmse_rf:.2f}")

# Evaluation: MAE
mae_rf = mae_eval.evaluate(rf_predictions)
print(f"Random Forest MAE: {mae_rf:.2f}")

# Evaluation: R2 Score
r2_rf = r2_eval.evaluate(rf_predictions)
print(f"Random Forest R2 Score: {r2_rf:.4f}")

Random Forest RMSE: 252.77
Random Forest MAE: 133.42
Random Forest R2 Score: 0.6726


### C. Deep Learning: Classification
Predict categorical high value status (`amount > 200`) using a Neural Network (Multilayer Perceptron).

In [100]:
print("\n[DL] Training Multilayer Perceptron Classifier (Neural Network)...")
# Layers: Input (1 feature) -> Hidden (5 nodes) -> Hidden (4 nodes) -> Output (2 classes)
layers = [1, 5, 4, 2]
# Block size 128 is reasonable for this data size
mlp = MultilayerPerceptronClassifier(layers=layers, blockSize=128, seed=42, 
                                     featuresCol="features", labelCol="high_value_label", maxIter=100)
mlp_model = mlp.fit(train_data)
mlp_predictions = mlp_model.transform(test_data)

class_eval = MulticlassClassificationEvaluator(labelCol="high_value_label", metricName="accuracy")
accuracy = class_eval.evaluate(mlp_predictions)
print(f"Neural Network Accuracy: {accuracy*100:.2f}%")


[DL] Training Multilayer Perceptron Classifier (Neural Network)...
Neural Network Accuracy: 98.76%


## 10. Save Results
Save the aggregated tables to CSV using Pandas.

In [101]:
# Use standard local path for Pandas (no file:// protocol)
output_path_cat = os.path.join(os.getcwd(), "output", "category_summary.csv")
output_path_trends = os.path.join(os.getcwd(), "output", "daily_trends.csv")
output_path_top_cust = os.path.join(os.getcwd(), "output", "top_customers.csv")

print(f"Saving summaries to output/...")

# Save Category Summary
category_summary.toPandas().to_csv(output_path_cat, index=False)

# Save Daily Trends
daily_trends.toPandas().to_csv(output_path_trends, index=False)

# Save Top Customers
top_customers.toPandas().to_csv(output_path_top_cust, index=False)

print("Pipeline completed successfully.")

Saving summaries to output/...
Pipeline completed successfully.


In [102]:
# Stop Spark Session when done
spark.stop()

*Dataset*: https://www.kaggle.com/datasets/umeshnaik01/sales_transactions. 

*REFERENCES* 
1. Munappy, A., Bosch, J., Holmström Olsson, H., et al. Modelling Data Pipelines. 
IEEE SEAA, 2020. 
https://ieeexplore.ieee.org/document/9226314. 

2. Yusuf, Y., & Venkat, A. Distributed Data Pipelines for Scalability in Predictive 
AI Frameworks. 
https://www.researchgate.net/profile/Charles-Paul8/publication/386440210_Distributed_Data_Pipelines_for_Scalability_in_Predictive_AI_Frameworks. 

3. Kandur, R. Building Distributed Data Processing Pipelines for AI Model 
Training. IRJMETS, 2025. 
https://www.irjmets.com/uploadedfiles/paper//issue_3_march_2025/68567/final/fin_irjmets1741206460.pdf 